In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/cs-training.csv")
df = df.rename(columns={"Unnamed: 0": "customer_id"})

target = "SeriousDlqin2yrs"

assert df.shape[0] == 150000, "数据行数不是150000，请检查文件"
assert df[target].isna().sum() == 0, "目标变量存在缺失，请检查数据"

df.loc[df["age"] == 0, "age"] = np.nan

delinquency_columns = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]

for column in delinquency_columns:
    df[column] = df[column].replace([96, 98], np.nan)

X = df.drop(columns=["customer_id", target])
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Training default rate:", y_train.mean())
print("Test default rate:", y_test.mean())

Training set: (120000, 10)
Test set: (30000, 10)
Training default rate: 0.06684166666666666
Test default rate: 0.06683333333333333


建立 Logistic Regression 模型

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor", Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler())
    ])),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42))
])

model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


输出风险概率

In [3]:
y_probability = model.predict_proba(X_test)[:, 1]

print("First 10 predicted probabilities:")
print(y_probability[:10])

First 10 predicted probabilities:
[0.02499458 0.022741   0.02671862 0.02930423 0.02602743 0.01437411
 0.02671369 0.46036613 0.0597998  0.07927491]


评价模型排序能力

ROC-AUC：模型区分高低风险的能力，越高越好。
Average Precision：在违约样本很少的情况下更有参考价值，越高越好。
Brier Score：概率准不准，越低越好。
Portfolio default rate：整体平均违约率[链接文字](https://)

In [5]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

roc_auc = roc_auc_score(y_test, y_probability)
average_precision = average_precision_score(y_test, y_probability)
brier = brier_score_loss(y_test, y_probability)

print("ROC-AUC:", roc_auc)
print("Average Precision:", average_precision)
print("Brier Score:", brier)
print("Portfolio default rate:", y_test.mean())

ROC-AUC: 0.8205032516048688
Average Precision: 0.3598188276486844
Brier Score: 0.052283412528698454
Portfolio default rate: 0.06683333333333333


做风险分层

In [6]:
prediction_results = pd.DataFrame({
    "actual_result": y_test.values,
    "predicted_probability": y_probability
})

prediction_results["risk_decile"] = (
    pd.qcut(
        prediction_results["predicted_probability"],
        q=10,
        labels=False,
        duplicates="drop"
    ) + 1
)

decile_summary = prediction_results.groupby("risk_decile").agg(
    customer_count=("actual_result", "count"),
    default_count=("actual_result", "sum"),
    observed_default_rate=("actual_result", "mean"),
    average_predicted_probability=("predicted_probability", "mean")
).reset_index()

decile_summary["observed_default_rate"] = decile_summary["observed_default_rate"] * 100
decile_summary["average_predicted_probability"] = decile_summary["average_predicted_probability"] * 100

decile_summary.sort_values("risk_decile", ascending=False)

,risk_decile,customer_count,default_count,observed_default_rate,average_predicted_probability
9,10,3000,1041,34.700000,29.633725
8,9,3000,286,9.533333,7.694794
7,8,3000,168,5.600000,5.984179
6,7,3000,129,4.300000,5.125029
5,6,3000,109,3.633333,4.406805
4,5,3000,83,2.766667,3.799327
3,4,3000,78,2.600000,3.252602
2,3,3000,49,1.633333,2.781679
1,2,3000,39,1.300000,2.340817
0,1,3000,23,0.766667,1.707523


## Baseline Logistic Regression Summary

This notebook builds the first baseline credit risk model using logistic regression.

The model outputs predicted default probabilities rather than making final approval or rejection decisions.

No final decision threshold is selected in this notebook. The purpose of this step is to evaluate whether the model can rank customers by credit risk.

Final threshold selection will be handled later in the model validation notebook.